# SHL Grammar Scoring: transcription and embeddings (Kaggle GPU)

This notebook does the heavy, GPU-only part of the pipeline **once** and saves everything the main notebook needs. Nothing here is trained; it only runs pretrained models.

| Stage | Model | Output file |
|---|---|---|
| 1. Transcribe every clip, plus simple timing stats | Whisper `medium.en` | `outputs/transcripts.csv` |
| 2. One embedding vector per transcript | MPNet sentence model, DeBERTa-v3-large | `outputs/emb_text_mpnet.npz`, `outputs/emb_text_deberta.npz` |
| 3. Grammatical acceptability of each sentence | RoBERTa fine-tuned on CoLA | `outputs/cola_features.csv` |
| 4. One embedding vector per clip from the raw audio | WavLM base plus | `outputs/emb_audio_wavlm.npz` |

**How to run it**
1. On the competition page open *Code > New Notebook* (this attaches the data), then *File > Import Notebook* and upload this file.
2. In the notebook settings set *Accelerator* to **GPU T4 x2** (only one GPU is used) and turn **Internet on** (needed to download the models).
3. *Save Version > Save & Run All (Commit)*. It takes roughly 30 to 45 minutes and keeps running if the browser is closed.
4. Open the finished version, go to its *Output* tab and download `shl_outputs.zip`. Unzip it in the repo root: it holds `outputs/` and `data/` (a copy of the three competition CSVs).

Every stage writes its own file as soon as it finishes and skips itself if that file already exists, so an interrupted interactive session can pick up where it stopped.

In [ ]:
import os
import random
import re
import shutil
import zipfile
from pathlib import Path

import librosa
import numpy as np
import pandas as pd
import torch
import transformers
from tqdm.auto import tqdm
from transformers import (AutoFeatureExtractor, AutoModel, AutoModelForSequenceClassification,
                          AutoModelForSpeechSeq2Seq, AutoProcessor, AutoTokenizer, pipeline)

# Pretrained models used below. All are public on the Hugging Face hub.
WHISPER_MODEL = "openai/whisper-medium.en"      # English-only Whisper: good accuracy, fast enough on a T4
TEXT_EMBEDDING_MODELS = {
    "mpnet": "sentence-transformers/all-mpnet-base-v2",  # trained to produce sentence embeddings
    "deberta": "microsoft/deberta-v3-large",               # general text encoder, strong on language-quality tasks
}
COLA_MODEL = "textattack/roberta-base-CoLA"     # classifies a sentence as grammatically acceptable or not
AUDIO_MODEL = "microsoft/wavlm-base-plus"       # self-supervised speech model, captures how the speech sounds

SAMPLE_RATE = 16_000  # every model above expects 16 kHz mono audio
SEED = 42

INPUT_DIR = Path("/kaggle/input")
WORK_DIR = Path("/kaggle/working")
OUT_DIR = WORK_DIR / "outputs"
DATA_COPY_DIR = WORK_DIR / "data"
OUT_DIR.mkdir(parents=True, exist_ok=True)
DATA_COPY_DIR.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
assert DEVICE == "cuda", "No GPU found: set Accelerator to GPU in the notebook settings"
print("torch", torch.__version__, "| transformers", transformers.__version__, "|", torch.cuda.get_device_name(0))

## 0. Find the data

Paths and column names are not hard-coded. The cell below lists every CSV and audio file under `/kaggle/input` and prints the CSV columns, so the layout is visible in the notebook output.

In [ ]:
def list_files(root, suffixes):
    # os.walk with followlinks=True also finds files behind symlinked input folders
    return sorted(Path(d) / f for d, _, names in os.walk(root, followlinks=True)
                  for f in names if Path(f).suffix.lower() in suffixes)

csv_paths = list_files(INPUT_DIR, {".csv"})
audio_paths = list_files(INPUT_DIR, {".wav"})
print(f"{len(csv_paths)} CSV files:")
for p in csv_paths:
    print("  ", p.relative_to(INPUT_DIR))
print(f"\n{len(audio_paths)} audio files, per folder:")
print(pd.Series([str(p.parent.relative_to(INPUT_DIR)) for p in audio_paths]).value_counts().to_string())

def find_csv(name):
    matches = [p for p in csv_paths if p.name.lower() == name]
    assert len(matches) == 1, f"expected exactly one {name}, found {matches}"
    return matches[0]

train_df = pd.read_csv(find_csv("train.csv"))
test_df = pd.read_csv(find_csv("test.csv"))
sample_sub = pd.read_csv(find_csv("sample_submission.csv"))
for name, df in [("train", train_df), ("test", test_df), ("sample_submission", sample_sub)]:
    print(f"\n{name}: {df.shape[0]} rows, columns {list(df.columns)}")
    print(df.head(3).to_string())

# Keep a copy of the CSVs next to the outputs so one download has everything the local notebook needs.
for name in ["train.csv", "test.csv", "sample_submission.csv"]:
    shutil.copy(find_csv(name), DATA_COPY_DIR / name)

In [ ]:
# Index every audio file by its name without extension ("audio_12.wav" -> "audio_12").
audio_by_stem = {}
for p in audio_paths:
    audio_by_stem.setdefault(p.stem, []).append(p)

def detect_file_column(df):
    # The file column is the one whose values match the audio file names.
    hit_rate = {c: df[c].astype(str).map(lambda v: Path(v).stem).isin(audio_by_stem).mean() for c in df.columns}
    best = max(hit_rate, key=hit_rate.get)
    assert hit_rate[best] > 0.99, f"no column matches the audio file names: {hit_rate}"
    return best

def resolve_audio(value, split):
    # If train and test reuse a file name, the folder name ("train"/"test") decides.
    candidates = audio_by_stem.get(Path(str(value)).stem, [])
    if len(candidates) > 1:
        candidates = [p for p in candidates
                      if any(split in part.lower() for part in p.relative_to(INPUT_DIR).parts[:-1])]
    assert len(candidates) == 1, f"cannot find a unique audio file for {value!r} ({split}): {candidates}"
    return candidates[0]

file_cols = {"train": detect_file_column(train_df), "test": detect_file_column(test_df)}
print("file name column:", file_cols)

# One row per clip, train first then test. Every output below keeps this order.
files = pd.concat([
    pd.DataFrame({"file_id": train_df[file_cols["train"]].astype(str), "split": "train"}),
    pd.DataFrame({"file_id": test_df[file_cols["test"]].astype(str), "split": "test"}),
], ignore_index=True)
files["path"] = [resolve_audio(f, s) for f, s in zip(files["file_id"], files["split"])]
print(files["split"].value_counts().to_string())

## 1. Transcription with Whisper, plus timing stats

Clips are 45 to 60 seconds, longer than Whisper's 30 second window, so the Hugging Face pipeline cuts each clip into overlapping 30 second chunks and stitches the text back together. Decoding is greedy, so the output is deterministic.

While each clip is in memory we also record three timing stats used later as fluency features: total duration, time with speech, and the number of pauses of half a second or more. A stretch counts as silence when it is more than 30 dB quieter than the loudest part of the clip.

In [ ]:
def load_audio(path):
    # 16 kHz mono float32, the input format all models here expect
    audio, _ = librosa.load(path, sr=SAMPLE_RATE, mono=True)
    return audio.astype(np.float32)

def timing_stats(audio):
    intervals = librosa.effects.split(audio, top_db=30)  # (start, end) samples of non-silent stretches
    gaps = (intervals[1:, 0] - intervals[:-1, 1]) / SAMPLE_RATE
    return {
        "duration_sec": len(audio) / SAMPLE_RATE,
        "voiced_sec": float(sum(end - start for start, end in intervals)) / SAMPLE_RATE,
        "n_long_pauses": int((gaps >= 0.5).sum()),
    }

TRANSCRIPTS_PATH = OUT_DIR / "transcripts.csv"

if TRANSCRIPTS_PATH.exists():
    transcripts = pd.read_csv(TRANSCRIPTS_PATH, dtype={"file_id": str})
    transcripts["transcript"] = transcripts["transcript"].fillna("")
    print("Loaded existing", TRANSCRIPTS_PATH)
else:
    processor = AutoProcessor.from_pretrained(WHISPER_MODEL)
    # Load, then cast to fp16 ourselves: half the memory and about twice as fast on a T4.
    whisper = AutoModelForSpeechSeq2Seq.from_pretrained(WHISPER_MODEL).half().to(DEVICE).eval()
    asr = pipeline("automatic-speech-recognition", model=whisper, tokenizer=processor.tokenizer,
                   feature_extractor=processor.feature_extractor, device=whisper.device,
                   chunk_length_s=30, batch_size=16)

    rows = []
    step = 32  # clips loaded per round, keeps RAM use small
    for start in tqdm(range(0, len(files), step), desc="Whisper"):
        part = files.iloc[start:start + step]
        audios = [load_audio(p) for p in part["path"]]
        outputs = asr(audios)
        for file_id, split, audio, out in zip(part["file_id"], part["split"], audios, outputs):
            rows.append({"file_id": file_id, "split": split, "transcript": out["text"].strip(), **timing_stats(audio)})

    transcripts = pd.DataFrame(rows)
    transcripts.to_csv(TRANSCRIPTS_PATH, index=False)
    del asr, whisper
    torch.cuda.empty_cache()

assert (transcripts["file_id"].values == files["file_id"].values).all()
assert (transcripts["split"].values == files["split"].values).all()

In [ ]:
# Quick sanity check: timing stats, empty transcripts, and the shortest and longest transcripts.
n_words = transcripts["transcript"].str.split().str.len()
print(transcripts[["duration_sec", "voiced_sec", "n_long_pauses"]].describe().round(2).to_string())
print("\nEmpty transcripts:", int((n_words == 0).sum()))
for label, idx in [("Shortest", n_words.nsmallest(2).index), ("Longest", n_words.nlargest(2).index)]:
    for i in idx:
        print(f"\n{label} ({n_words[i]} words, {transcripts.at[i, 'file_id']}):\n{transcripts.at[i, 'transcript']}")

## 2. Text embeddings

Each transcript goes through a pretrained text encoder, and the token vectors are averaged (ignoring padding) into one fixed-size vector per transcript. These vectors capture vocabulary, phrasing and fluency of the text, and a linear model on top of them learns which directions go with higher grammar scores.

Two encoders are saved so the main notebook can compare them with cross-validation and keep the better one.

In [ ]:
texts = transcripts["transcript"].tolist()

def save_npz(path, emb):
    # Store the row keys with the matrix so the main notebook can verify the order.
    np.savez_compressed(path, file_id=transcripts["file_id"].to_numpy(dtype=str),
                        split=transcripts["split"].to_numpy(dtype=str), emb=emb)

def mean_pooled_embeddings(model_name, batch_size=16, max_length=512):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name).to(DEVICE).eval()
    vectors = []
    for start in tqdm(range(0, len(texts), batch_size), desc=model_name):
        batch = tokenizer(texts[start:start + batch_size], padding=True, truncation=True,
                          max_length=max_length, return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            hidden = model(**batch).last_hidden_state
        mask = batch["attention_mask"].unsqueeze(-1).to(hidden.dtype)
        vectors.append(((hidden * mask).sum(1) / mask.sum(1).clamp(min=1)).float().cpu().numpy())
    del model
    torch.cuda.empty_cache()
    return np.vstack(vectors)

for short_name, model_name in TEXT_EMBEDDING_MODELS.items():
    path = OUT_DIR / f"emb_text_{short_name}.npz"
    if path.exists():
        print("Already saved:", path.name)
        continue
    try:
        emb = mean_pooled_embeddings(model_name)
        save_npz(path, emb)
        print(path.name, emb.shape)
    except Exception as err:  # later stages do not depend on this one, so keep going
        print(f"FAILED {model_name}: {err!r}")

## 3. Sentence acceptability (CoLA)

The Corpus of Linguistic Acceptability (CoLA) is a set of English sentences labelled as grammatical or not. A RoBERTa model fine-tuned on it gives, for each sentence, the probability that it is grammatically acceptable. We split each transcript into sentences using Whisper's punctuation, score every sentence, and keep three summaries per clip: the mean probability, the lowest one, and the share of sentences below 0.5.

Rather than trusting the model's label names, a few obviously good and bad sentences are used to check which output means "acceptable".

In [ ]:
COLA_PATH = OUT_DIR / "cola_features.csv"

def split_sentences(text):
    return [s for s in re.split(r"(?<=[.!?])\s+", text.strip()) if s]

if COLA_PATH.exists():
    print("Already saved:", COLA_PATH.name)
else:
    try:
        tokenizer = AutoTokenizer.from_pretrained(COLA_MODEL)
        model = AutoModelForSequenceClassification.from_pretrained(COLA_MODEL).to(DEVICE).eval()

        def class_probs(sentences):
            batch = tokenizer(sentences, padding=True, truncation=True, max_length=128, return_tensors="pt").to(DEVICE)
            with torch.no_grad():
                return torch.softmax(model(**batch).logits, dim=-1).cpu().numpy()

        probe = class_probs(["The cat is sleeping on the sofa.", "She has already finished her homework.",
                             "Cat the sofa on sleeping is the.", "She have finish homework her yesterday already."])
        ok_idx = int(np.argmax(probe[:2].mean(0) - probe[2:].mean(0)))
        print("Probe probabilities (2 good, 2 bad):\n", probe.round(3), "\nAcceptable = output", ok_idx)

        sentence_rows = [(row, s) for row, text in enumerate(texts) for s in split_sentences(text)]
        sentences = [s for _, s in sentence_rows]
        p_ok = np.concatenate([class_probs(sentences[i:i + 64])[:, ok_idx]
                               for i in tqdm(range(0, len(sentences), 64), desc="CoLA")])

        per_sentence = pd.DataFrame({"row": [r for r, _ in sentence_rows], "p_ok": p_ok})
        summary = per_sentence.groupby("row")["p_ok"].agg(
            cola_mean="mean", cola_min="min", cola_frac_bad=lambda p: float((p < 0.5).mean()))
        cola = transcripts[["file_id", "split"]].join(summary)  # clips with no sentences get NaN
        cola.to_csv(COLA_PATH, index=False)
        print(cola.describe().round(3).to_string())
        del model
        torch.cuda.empty_cache()
    except Exception as err:
        print(f"FAILED {COLA_MODEL}: {err!r}")

## 4. Audio embeddings (WavLM)

The text pipeline cannot hear pronunciation, rhythm or hesitation that Whisper smooths over. WavLM is a speech model trained on raw audio without labels; it turns every 20 ms of audio into a vector at each of its 13 layers. Averaging over time gives one 13 x 768 matrix per clip. All layers are saved (as float16 to keep the file small) and the main notebook averages them.

In [ ]:
WAVLM_PATH = OUT_DIR / "emb_audio_wavlm.npz"

if WAVLM_PATH.exists():
    print("Already saved:", WAVLM_PATH.name)
else:
    try:
        extractor = AutoFeatureExtractor.from_pretrained(AUDIO_MODEL)
        model = AutoModel.from_pretrained(AUDIO_MODEL).to(DEVICE).eval()
        layers = []
        for path in tqdm(files["path"], desc="WavLM"):
            inputs = extractor(load_audio(path), sampling_rate=SAMPLE_RATE, return_tensors="pt").to(DEVICE)
            with torch.no_grad():
                hidden = model(**inputs, output_hidden_states=True).hidden_states
            layers.append(torch.stack([h[0].mean(0) for h in hidden]).cpu().numpy().astype(np.float16))
        emb = np.stack(layers)
        save_npz(WAVLM_PATH, emb)
        print(WAVLM_PATH.name, emb.shape)
        del model
        torch.cuda.empty_cache()
    except Exception as err:
        print(f"FAILED {AUDIO_MODEL}: {err!r}")

## 5. Bundle the outputs

Everything goes into one zip so a single download from the *Output* tab is enough. Unzip it in the repo root.

In [ ]:
bundle = WORK_DIR / "shl_outputs.zip"
with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as zf:
    for folder in [OUT_DIR, DATA_COPY_DIR]:
        for p in sorted(folder.iterdir()):
            zf.write(p, p.relative_to(WORK_DIR))
            print(f"{str(p.relative_to(WORK_DIR)):40s} {p.stat().st_size / 1e6:7.2f} MB")
print(f"\n{bundle.name}: {bundle.stat().st_size / 1e6:.1f} MB")